In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer,pipeline
from huggingface_hub import login
from sentence_transformers import SentenceTransformer
login()

model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-3B-Instruct",
    device_map="cuda",
    torch_dtype="auto",use_cache=True
)
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cuda"  )

print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())

In [ ]:
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=200,
    do_sample=False,
)


In [ ]:
messages = [
    {"role": "user", "content": "Create a funny joke about chickens."}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)

In [ ]:
def generate(messages, max_new_tokens=200, do_sample=True, temperature=1, top_p=0.25):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [ ]:
messages = [{"role": "user", "content": "Classify the text into neutral, negative or positive.\nText: I think the food was okay.\nSentiment:"}]
print(generate(messages))

In [ ]:
persona = "You are an expert in AI programming assistant.Help solving, writing, explaining any code to make the user's work easy.\n"
instruction= "Give a step by step answer for the user's question.If it's a coding, answer should be working code.\n"
context = "The assistance is used by the some developers.\n"
data_format = """1. Question summery.
2. Explaination
3. Code (if applicabel)
4. Conclusion\n"""
audience = "The user is beginner to intermediate programer.\n"
tone = "Friendly, professional, and concise.\n"
data = "Explain me a C program for sum of two numbers in a simple way with code."
full_prompt = persona + instruction + context + data_format + audience + tone + data
messages = [{"role": "user", "content": full_prompt}]
print(generate(messages,max_new_tokens=350))